In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
 
from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

# for working with .toml
import tomli_w

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return ""
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return ""

In [ ]:
# 1. Load the environment variables
load_dotenv("APA_localization.scicore.env")

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "UCSCtracks_dir": os.getenv("UCSC_TRACKS_DIR"),
    "UCSCtracks_trackfiles_dir": os.getenv("UCSC_TRACKFILES_DIR"),
    "UCSCtracks_trackhubs_dir": os.getenv("UCSC_TRACKHUBS_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "mouse_annotation_dir": os.getenv("MOUSE_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "external_data_dir": os.getenv("EXTERNAL_DATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "pod5_dir": os.getenv("POD5_DIR"), # nanopore-specific
    "dorado_models_dir": os.getenv("DORADO_MODELS_DIR"), # nanopore-specific
    "nanoflowz_dir": os.getenv("NANOFLOWZ_DIR"), # nanopore-specific
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_chrom_sizes_file": os.getenv("HUMAN_CHROM_SIZES_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
    "human_basic_annotation_file": os.getenv("HUMAN_BASIC_ANNOTATION_FILE"),
    "human_polyAsite_atlas": os.getenv("HUMAN_POLYASITE_ATLAS"),
    "human_tandem_PAS": os.getenv("HUMAN_TANDEM_PAS"),
    "human_exonic_segments_gtf": os.getenv("HUMAN_EXONIC_SEGMENTS_GTF"),
    "human_exonic_segments_bed": os.getenv("HUMAN_EXONIC_SEGMENTS_BED"),
    "dorado_executor": os.getenv("DORADO_EXECUTOR"), # nanopore-specific
    "dorado_executor_v2_0": os.getenv("DORADO_EXECUTOR_v2_0"), # nanopore-specific
    "dorado_executor_v2_1": os.getenv("DORADO_EXECUTOR_v2_1"), # nanopore-specific
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Prepare start samples for alignment

In [ ]:
# we extract the paths to .pod5 files

os.system("""find """+subdirs['pod5_dir']+'p40713_o42416_1/'+""" -name '*.pod5' > """+subdirs['temp_dir']+"""CD47_pod5_files.tsv""")

In [ ]:
# so far don't filter out reads marked as "fail". We'll try afterwards if needed

In [ ]:
pod5_file_paths = pd.read_csv(subdirs['temp_dir']+'CD47_pod5_files.tsv',delimiter="\t",
                                   index_col=None,header=None)
pod5_file_paths.columns = ['pod5']
pod5_file_paths['sample_id'] = pod5_file_paths.apply(lambda x:x['pod5'].split('/')[-2],1)
pod5_file_paths = pod5_file_paths.loc[pod5_file_paths['pod5'].str.contains('pass')].reset_index(drop=True) # only look at reads marked as "pass"

In [ ]:
len(pod5_file_paths)

In [ ]:
# we are only interested in barcodes 13-24
sel_sample_ids = range(13,25)
sel_sample_ids = [('barcode0'+str(elem) if len(str(elem))==1 else 'barcode'+str(elem)) for elem in sel_sample_ids]
pod5_file_paths = pod5_file_paths.loc[pod5_file_paths['sample_id'].isin(sel_sample_ids)].reset_index(drop=True)

In [ ]:
pod5_file_paths['sample_id'].unique()

In [ ]:
WF_version = 'v2p1_ONT_CD47_v1' # v2p0 at the beginning indicated Dorado version, while v2/v3 at the end indicates the particular version of the analysis

dir_path = subdirs['wf_runs_dir']+WF_version+'/'
(Path(dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory

pod5_file_paths[['sample_id','pod5']].to_csv(dir_path+'start_samples.tsv', sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

# Configuring nanoflowz execution

We've created a conda env "nextflow" to execute nanoflowz:

```bash
conda create --name nextflow bioconda::nextflow
conda activate nextflow
```

## v2p1_ONT_CD47_v1

In [ ]:
WF_version = 'v2p1_ONT_CD47_v1'
dir_path = subdirs['wf_runs_dir']+WF_version+'/'

run_dir = subdirs['wf_runs_dir']+WF_version+'/'

# create a .toml file for polyA tail profiling
polyA_toml_output_path = os.path.join(dir_path,'polyA_config.toml')
(Path(polyA_toml_output_path).parent).mkdir(parents=True, exist_ok=True) # create subdir, just in case

config_data = {
    "tail": {
        "tail_interrupt_length": 1
    }
}
with open(polyA_toml_output_path, "wb") as f:
    tomli_w.dump(config_data, f)

# define pipeline configuration
json_output_path = dir_path+'run_params.json'
json_file_dir = Path(json_output_path).parent
json_file_dir.mkdir(parents=True, exist_ok=True)

# define custom parameters for nanoflowz run

json_params_content = {
    "tsv": str(Path(dir_path+'start_samples.tsv').resolve()),
    "rundir": str(Path(run_dir).resolve()),
    "dorado": str(file_paths['dorado_executor_v2_1']),
    "model": str(os.path.join(subdirs['dorado_models_dir'], 'dna_r10.4.1_e8.2_400bps_sup@v5.2.0')),
    "polyA": str(polyA_toml_output_path),
    "ref": str(file_paths['human_genome_file']),
    "reference_gtf": str(file_paths['human_basic_annotation_file']),
    "shift_ambiguous_cs": False, # <- here we don't enable the shifting of cleavage site positions!
    "check_gtf_compatibility": False, # no need, there are too many scaffolds in the genome that are not present in the used basic annotation
}
    
params_file = Path(json_output_path)
    
with open(params_file, "w") as f:
    json.dump(json_params_content, f, indent=4)

# run the printed command from any directory under the LOGIN node
# note that it also includes '-resume' argument to make sure that by default it won't be executed from the beginning
cmd = 'nextflow run '+\
subdirs['nanoflowz_dir']+'main.nf'+\
' -params-file '+str(params_file.resolve())+\
' -profile conda'+\
' -resume'
print(cmd)

# Define metadata labels and colors uniformly

In [ ]:
def get_metadata_with_labels_and_colors(input_df):
    sel_files = input_df.copy()
    condiction_dict = {
        13:'WT_neg_short',
        14:'WT_neg_short',
        15:'WT_neg_short',
        16:'WT_neg_long',
        17:'WT_neg_long',
        18:'WT_neg_long',
        19:'short_excision',
        20:'short_excision',
        21:'short_excision',
        22:'long_excision',
        23:'long_excision',
        24:'long_excision'
    }
    sel_files["condition"] = sel_files['Barcode number'].map(condiction_dict)

    sel_files["sample_label_within_experiment"] = (
        sel_files["condition"].astype('str')+ ";"
        + sel_files["Barcode number"].astype('str')
    )

    cat = "condition"
    cat_order = ["WT_neg_short", "WT_neg_long", "short_excision","long_excision"]
    cat_palette = [
        "grey",
        "teal",
        "royalblue",
        "orangered",
    ]
    cat_color_dict = {}
    for k, cat_val in enumerate(cat_order):
        cat_color_dict[cat_val] = cat_palette[k]
    sel_files[cat + ";color"] = sel_files[cat].map(cat_color_dict)

    sel_files = sel_files.sort_values(
        ["condition", 'Barcode number', "sample"]
    ).reset_index(drop=True)
    return sel_files, cat_order, cat_palette

# PolyA-tail lengths vs abundance vs CPA analysis - on gene level

In [ ]:
WF_version = "v2p1_ONT_CD47_v1"
organism = "human"

command = 'find '+os.path.join(subdirs['wf_runs_dir'],WF_version,'results','read_tags')+\
        " -name '*.tags.tsv.gz' > "+\
          os.path.join(subdirs['temp_dir'],"read_tag_tables.files."+WF_version+".txt")
out = subprocess.check_output(command, shell=True)

read_tag_tables_files = pd.read_csv(os.path.join(subdirs['temp_dir'],"read_tag_tables.files."+WF_version+".txt"),delimiter="\t",
                                   index_col=None,header=None)

entity_col = "sample" # should be a unique index for the metadata table
read_tag_tables_files[entity_col] = read_tag_tables_files.apply(lambda x:x[0].split('/')[-1].replace('.tags.tsv.gz',''),1)
read_tag_tables_files = read_tag_tables_files.rename(columns={0:'path'})
read_tag_tables_files['Barcode number'] = read_tag_tables_files.apply(lambda x:int(x['sample'].replace('barcode','')),1)

barcodes_metadata_df = read_tag_tables_files.copy() # for consistency of namings
cat_name = "condition"
metadata_df, cat_order, cat_palette = get_metadata_with_labels_and_colors(barcodes_metadata_df.copy())


In [ ]:
# select chromosomes at which we want to look
chromosomes_pd = pd.read_csv(file_paths['human_basic_annotation_file'],delimiter="\t",
                                   index_col=None,header=None,usecols=[0])
chromosomes_pd = chromosomes_pd.drop_duplicates().reset_index(drop=True)

sel_chromosome_list = list(chromosomes_pd.loc[(chromosomes_pd[0].str.startswith('chr'))][0])

def natural_sort_key(s):
    parsed = re.split('([0-9]+)', str(s))
    return [(0, int(text)) if text.isdigit() else (1, text.lower()) for text in parsed]

sel_chromosome_list = sorted(sel_chromosome_list, key=natural_sort_key)

In [ ]:
tmp.head()

In [ ]:
i=0
res_dict = {}

for index,row in metadata_df.iterrows():
    tmp = pd.read_csv(row['path'],delimiter="\t",index_col=None,header=0,usecols=[1,2,3,4,5])
    cols = list(tmp.columns)

    tmp = tmp.loc[tmp['chrom'].isin(sel_chromosome_list)].reset_index(drop=True)
    tmp['chrom'] = pd.Categorical(tmp['chrom'], categories=sel_chromosome_list, ordered=True)

    # only look at rows where pt was estimated
    tmp = tmp.loc[tmp['pt']>0]
    
    # calculate median-per-gene and abundance
    tmp['w'] = 1
    tmp['w_pt'] = tmp['w']*tmp['pt'] # we will calculate weighted mean
    
    index_cols = ['chrom','XT']
    gr = tmp.groupby(index_cols).agg({'w':np.sum,'w_pt':np.sum}).reset_index()
    gr['pt_av'] = gr['w_pt']/gr['w']
    
    # we'll make a sample matrix for average polyA tail lengths and for abundance
    features = ['pt_av','w']
    for feature in features:
        if i==0:
            res_dict[feature] = gr[index_cols+[feature]].rename(columns={feature:row[entity_col]}).copy()
        else:
            res_dict[feature] = pd.merge(res_dict[feature],gr[index_cols+[feature]].rename(columns={feature:row[entity_col]}),
                                         how='outer',on=index_cols)
    if (i+1)%2==0:
        print(str(i)+' done')
    i=i+1

In [ ]:
sel_metadata = metadata_df.copy()
samples_list = list(sel_metadata['sample'])

In [ ]:
res_dict['w'][samples_list] = np.round(res_dict['w'][samples_list].fillna(0)).astype(int) # make integer counts

In [ ]:
l = res_dict['w'].loc[~res_dict['w']['XT'].str.contains(',')][samples_list].sum()/10**6
l.sort_values()

### Sanity style analysis

In [ ]:
# run gtf parsing
from zavolab_pyutils import (
    parse_gtf_attributes_into_pd_dataframes,
)

gtf_df, genes_df, exons_df = parse_gtf_attributes_into_pd_dataframes(file_paths['human_basic_annotation_file'],
                                                                     gene_type_field = "gene_type",
                                                                     extract_exon_number = True, 
                                                                     extract_gene_name_in_exons = True,
                                                                     verbose=False,
                                                                    )

In [ ]:
from zavolab_pyutils import (
    apply_deseq2_normalization, get_MultiDimR2,
    prepare_isoform_sanity_matrix, 
    apply_sanity_normalization_full_bayesian, 
    test_differential_relative_usage,
    test_differential_expression
)

In [ ]:
sel_metadata = metadata_df.copy() # we'll try now to include everything

# adhere to sort order defined for cat_name
sel_metadata[cat_name] = pd.Categorical(sel_metadata[cat_name],categories=cat_order,ordered=True)
sel_metadata = sel_metadata.sort_values([cat_name,'sample']).reset_index(drop=True)

samples_list = list(sel_metadata['sample'])

In [ ]:
# we perform Sanity normalization and variance estimation
raw_counts_df = res_dict['w'].copy()
raw_counts_df.index = raw_counts_df['XT'].values

# Sanity-normalized counts are already log2-transformed
sanity_norm_counts_df, sanity_means_df, sanity_relative_errors_df, sanity_absolute_errors_df, sanity_vg_df, median_lib_size, variances_df = apply_sanity_normalization_full_bayesian(
    counts_df=raw_counts_df, 
    metadata_df=sel_metadata.copy(), 
    sample_col='sample', 
    cond_col='condition',
    vmin=0.0000001, 
    vmax=100,
    n_cores=12,
    empirical_bayes=False, # < Important!
    loess_variance_threshold_q=0.15,
)

In [ ]:
sel_metadata

In [ ]:
# Diagnostic plots for Sanity outputs

from zavolab_pyutils import (
    plot_variance_vs_expression,
    plot_mean_vs_cv,
)

# discard the effect from library size, as was done in original Sanity paper
ToCompare_sanity_norm_counts_df = sanity_norm_counts_df-np.log2(median_lib_size)

sanity_vg_df['inferred_v_g'] = sanity_vg_df['MAP_v_g'] # use MAP estimate for Vg to plot, in-line with original Sanity implementation
plot_variance_vs_expression(
    ToCompare_sanity_norm_counts_df, sanity_vg_df,
    savefig_path=subdirs['figures_dir']+'gene_expression_analysis/CD47_pgRNAexcision/pySanity/variance_vs_expr.ONT.png',
    true_vg=None, # This is to compare with True value specified during simulation above
    ylim = (0,10),
)

natScale_ToCompare_sanity_norm_counts_df = 2**ToCompare_sanity_norm_counts_df

sanity_plot_data = plot_mean_vs_cv(
natScale_ToCompare_sanity_norm_counts_df, sel_metadata.copy(),
savefig_path=subdirs['figures_dir']+'gene_expression_analysis/CD47_pgRNAexcision/pySanity/cv_vs_mean_plot.ONT.png')

In [ ]:
# Run PCA - now based on Sanity normalized counts

from zavolab_pyutils import (
    pca_plot
)

sanity_norm_counts_df['m'] = sanity_norm_counts_df[samples_list].mean(axis=1)
# PCA_UMAP_subset = sanity_norm_counts_df.loc[sanity_norm_counts_df['m']>sanity_norm_counts_df['m'].quantile(0.1)].copy().reset_index(drop=True)

expressed_everywhere_genes = list(raw_counts_df.loc[raw_counts_df[samples_list].min(axis=1)>0].index) # require at least one read in all samples!
PCA_UMAP_subset = sanity_norm_counts_df.loc[expressed_everywhere_genes].copy().reset_index(drop=True) 

print(len(PCA_UMAP_subset))

savefig_path = (
    subdirs['figures_dir']+'gene_expression_analysis/CD47_pgRNAexcision/pySanity/PCA_gene_expression.Sanity_norm_counts.ONT.min1read.png'
)

hue_column = cat_name
palette = cat_palette
hue_order = cat_order

pca_plot(
    PCA_UMAP_subset,
    samples_list,
    sel_metadata,
    "condition",
    savefig_path,
    sns_color_palette = palette,
    hue_order = hue_order,
    calculate_permanova_R2=True,
    permanova_R2_ajusted = False, # not adjusted for number of features, otherwise get negative values here
    add_text_labels_for_samples = True,
    text_label_column_for_samples = "Barcode number",
    s_param=40,
    figsize=(4.2,4.2),
)

# we also save tsv-formatted data frames that were used to generate plots alongside with the figures
save_tsv_path = str(Path(savefig_path).with_suffix('.tsv'))
PCA_UMAP_subset.to_csv(save_tsv_path,sep=str("\t"),header=True,index=None,quoting=csv.QUOTE_NONE,)

In [ ]:
# test excision of CD47short PAS
sanity_DE_df = test_differential_expression(
        means_df=sanity_means_df, 
        errors_df=sanity_relative_errors_df, # do not include global uncertainty to be a bit less conservative
        cond_A="short_excision", # numerator in FC
        cond_B="WT_neg_short"
    )
sanity_DE_df['gene_id'] = sanity_DE_df.index
sanity_DE_df = pd.merge(genes_df[['gene_id','gene_name','gene_type']],sanity_DE_df,how='right',on=['gene_id'])
sanity_DE_df = sanity_DE_df.sort_values('p_value').reset_index(drop=True)

In [ ]:
out_dir_path = os.path.join(subdirs['tables_dir'],'gene_expression_analysis','CD47_pgRNAexcision','pySanity','diff_expression')
(Path(out_dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory
sanity_DE_df.to_csv(
    os.path.join(out_dir_path,'pgRNAshort_vs_WT.tsv'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)

In [ ]:
sanity_DE_df.loc[sanity_DE_df['padj']<0.05]

In [ ]:
out_dir_path = os.path.join(subdirs['tables_dir'],'gene_expression_analysis','CD47_pgRNAexcision','pySanity','for_GO','pgRNAshort_vs_WT')
(Path(out_dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory

sanity_DE_df.loc[(sanity_DE_df['padj']<0.05)&(sanity_DE_df['log2FC']>0)&(
    ~sanity_DE_df['gene_name'].isna())][['gene_name']].to_csv(
    os.path.join(out_dir_path,'significantly_upregulated.txt'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)
sanity_DE_df.loc[(sanity_DE_df['padj']<0.05)&(sanity_DE_df['log2FC']<0)&(
    ~sanity_DE_df['gene_name'].isna())][['gene_name']].to_csv(
    os.path.join(out_dir_path,'significantly_downregulated.txt'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)
sanity_DE_df.loc[(~sanity_DE_df['gene_name'].isna())][['gene_name']].to_csv(
    os.path.join(out_dir_path,'background.txt'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)

In [ ]:
# test pgRNA long vs CTRl - retained weird naming "sanity_DE_df_OE" inherited from other project
sanity_DE_df_OE = test_differential_expression(
        means_df=sanity_means_df, 
        errors_df=sanity_relative_errors_df, # do not include global uncertainty to be a bit less conservative
        cond_A="long_excision", # numerator in FC
        cond_B="WT_neg_long"
    )
sanity_DE_df_OE['gene_id'] = sanity_DE_df_OE.index
sanity_DE_df_OE = pd.merge(genes_df[[0,'gene_id','gene_name','gene_type']].rename(columns = {0:'chrom'}),sanity_DE_df_OE,how='right',on=['gene_id'])
sanity_DE_df_OE = sanity_DE_df_OE.sort_values('p_value').reset_index(drop=True)

In [ ]:
out_dir_path = os.path.join(subdirs['tables_dir'],'gene_expression_analysis','CD47_pgRNAexcision','pySanity','diff_expression')
(Path(out_dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory
sanity_DE_df_OE.to_csv(
    os.path.join(out_dir_path,'pgRNAlong_vs_WT.tsv'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)

In [ ]:
out_dir_path = os.path.join(subdirs['tables_dir'],'gene_expression_analysis','CD47_pgRNAexcision','pySanity')
(Path(out_dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory
sanity_norm_counts_df.to_csv(
    os.path.join(out_dir_path,'pySanity_normalized_gene_counts.tsv'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)

In [ ]:
len(sanity_DE_df),len(sanity_DE_df_OE)

In [ ]:
tmp = sanity_norm_counts_df.copy()
if 'm' in sanity_DE_df.columns:
    sanity_DE_df = sanity_DE_df.drop(['m'],axis=1)
l = list(sel_metadata.loc[sel_metadata['condition'].isin(['short_excision','WT_neg_short'])]['sample'])
tmp['m'] = tmp[l].mean(axis=1) # calculate mean only for selected samples
sanity_DE_df = pd.merge(sanity_DE_df,tmp[['m']],how='left',left_on = 'gene_id',right_index=True)

if 'm' in sanity_DE_df_OE.columns:
    sanity_DE_df_OE = sanity_DE_df_OE.drop(['m'],axis=1)
l = list(sel_metadata.loc[sel_metadata['condition'].isin(['long_excision','WT_neg_long'])]['sample'])
tmp['m'] = tmp[l].mean(axis=1) # calculate mean only for selected samples
sanity_DE_df_OE = pd.merge(sanity_DE_df_OE,tmp[['m']],how='left',left_on = 'gene_id',right_index=True)

In [ ]:
len(sanity_DE_df),len(sanity_DE_df_OE)

In [ ]:
sanity_DE_df_OE.loc[sanity_DE_df_OE['padj']<0.05]

In [ ]:
sanity_DE_df_OE.loc[sanity_DE_df_OE['gene_name']=='CD47']

In [ ]:
sel_metadata

In [ ]:
# for KD
sanity_DE_df['DE_test'] = (sanity_DE_df['padj']<0.05).astype('int')*((sanity_DE_df['log2FC']>0).astype('int')*2-1)
sanity_DE_df['DE_test'] = sanity_DE_df['DE_test'].map({0:'NS',1:'up',-1:'down'})

map_dict = {}
l = list(sanity_DE_df['DE_test'].unique())
l.sort()
hue_order = []
for val in l:
    map_dict[val] = val+'; '+str(len(sanity_DE_df.loc[sanity_DE_df['DE_test']==val]))+' genes'
    hue_order.append(map_dict[val])
sanity_DE_df['DE_test'] = sanity_DE_df['DE_test'].map(map_dict)
palette = ['grey','green','red']

sns.set(font_scale=1, style="white")
fig, axes = plt.subplots(1, 1, figsize=(4.2, 4.2))

x_feature,y_feature,hue = 'm','log2FC','DE_test'

ax = sns.scatterplot(data = sanity_DE_df,x=x_feature,y=y_feature,s=5,hue=hue,hue_order = hue_order,palette=palette,alpha=0.6)
ax.set(xlabel = 'mean gene expression, $log_2$',ylabel = '$log_2$FC',ylim = (-8,8))
ax.tick_params(left=True, bottom=True)
ax.legend(
    bbox_to_anchor=(1.05, 1),
    loc=2,
    borderaxespad=0.0,
    title='short_excision -vs- WT_neg_short',
    markerscale=2,
    ncol=1,
)
savefig_path = (
    subdirs['figures_dir']+'gene_expression_analysis/CD47_pgRNAexcision/pySanity/MA_plot.pgRNAshort_vs_CTRL.pySanity.png'
)

fig.savefig(
        savefig_path,
        bbox_inches="tight",
        dpi=600,
    )

# for OE

sanity_DE_df_OE['DE_test'] = (sanity_DE_df_OE['padj']<0.05).astype('int')*((sanity_DE_df_OE['log2FC']>0).astype('int')*2-1)
sanity_DE_df_OE['DE_test'] = sanity_DE_df_OE['DE_test'].map({0:'NS',1:'up',-1:'down'})

map_dict = {}
l = list(sanity_DE_df_OE['DE_test'].unique())
l.sort()
hue_order = []
for val in l:
    map_dict[val] = val+'; '+str(len(sanity_DE_df_OE.loc[sanity_DE_df_OE['DE_test']==val]))+' genes'
    hue_order.append(map_dict[val])
sanity_DE_df_OE['DE_test'] = sanity_DE_df_OE['DE_test'].map(map_dict)
palette = ['grey','green','red']

sns.set(font_scale=1, style="white")
fig, axes = plt.subplots(1, 1, figsize=(4.2, 4.2))

x_feature,y_feature,hue = 'm','log2FC','DE_test'

ax = sns.scatterplot(data = sanity_DE_df_OE,x=x_feature,y=y_feature,s=5,hue=hue,hue_order = hue_order,palette=palette,alpha=0.6)
ax.set(xlabel = 'mean gene expression, $log_2$',ylabel = '$log_2$FC',ylim = (-8,8))
ax.tick_params(left=True, bottom=True)
ax.legend(
    bbox_to_anchor=(1.05, 1),
    loc=2,
    borderaxespad=0.0,
    title='long_excision -vs- WT_neg_long',
    markerscale=2,
    ncol=1,
)
savefig_path = (
    subdirs['figures_dir']+'gene_expression_analysis/CD47_pgRNAexcision/pySanity/MA_plot.pgRNAlong_vs_CTRL.pySanity.png'
)

fig.savefig(
        savefig_path,
        bbox_inches="tight",
        dpi=600,
    )

In [ ]:
# plot expression of selected genes across conditions along with its 95% confidence intervals (multiple testing bonferroni-adjusted)
# here, focus on house-keeping genes and genes involved in NPC complex and PAIP1 interactions
from zavolab_pyutils import (
    plot_sanity_gene_expression_with_ci,
)

# from annotation
# sel_gene_names = ['GAPDH','ACTR1B','PAIP1','PAIP1_isoform1_gene','Hygromycin_resistance_gene','Amp_resistance_gene']
sel_gene_names = ['GAPDH','ACTR1B','CD47']

# to define the order
sel_genes_df = genes_df.loc[genes_df['gene_name'].isin(sel_gene_names)].reset_index(drop=True).copy()
sel_genes_df['gene_name'] = pd.Categorical(sel_genes_df['gene_name'], categories=sel_gene_names, ordered=True)
sel_genes_df = sel_genes_df.sort_values('gene_name')
selected_genes = list(sel_genes_df['gene_id'])

order = cat_order
palette = cat_palette

plot_sanity_gene_expression_with_ci(
    sample_norm_df=sanity_norm_counts_df, 
    means_df=sanity_means_df, 
    errors_df=sanity_relative_errors_df, # we'll use relative errors (a bit smaller than absolute errors)
    metadata_df=sel_metadata.copy(), 
    selected_genes=selected_genes, 
    adjust_multiple_comparisons=True,
    savefig_path=subdirs['figures_dir']+'gene_expression_analysis/CD47_pgRNAexcision/pySanity/over_genes/'+'_'.join(sel_gene_names)+'.expression.png',
    genes_df=genes_df.copy(),
    condition_order = order,
    palette = palette,
    subplot_height = 3.2,
)